# 策略概述

**K-means** 是命題 1 中第三種機器學習分群方法，與 HDBSCAN（密度式）、
Agglomerative（階層式）構成三種**聚類典範**的對照。

三者使用**完全相同的混合特徵矩陣**（價格因子 ⊕ PIT 基本面 ⊕ 產業），
差異僅在如何從該特徵空間切出配對搜尋空間：

| 分群法 | 典範 | 群數決定 | 噪音處理 |
| :--- | :--- | :--- | :--- |
| HDBSCAN | 密度式 | 自動（由密度結構決定） | 低密度點標記為噪音並排除 |
| Agglomerative | 階層式 | 由合併距離分位數切割 | 過小群併入「未分類」 |
| **K-means** | **分割式** | **需外部指定 $k$** | **無——每點必屬某群** |

K-means 的加入使命題 1 的分群維度涵蓋三大典範，而非僅比較兩種相似方法。


# 群數 $k$ 如何決定：對齊 Agglomerative

K-means 需事先指定 $k$，而 HDBSCAN 與 Agglomerative 的群數由資料決定。
若對 K-means 另行以肘部法或輪廓係數選 $k$，將引入**第二個變動來源**——
分群結果的差異會同時來自「演算法」與「群數」，違反單變因對照的設計。

因此本策略每個形成窗**先跑一次 Agglomerative 取得其群數 $n$，再令 $k = n$**：

$$k_t = \big|\{\text{Agglomerative 於窗 } t \text{ 產生的群}\}\big|$$

如此三種分群法在每個窗口的**分組粒度可比**，
矩陣中觀察到的差異可歸因於分群典範本身，而非群數設定。

> 代價：$k$ 並非對 K-means 最佳化的結果，故本策略的表現應理解為
> 「在與 Agglomerative 相同粒度下，分割式聚類的表現」，
> 而非「K-means 所能達到的最佳表現」。


# 參考文獻與引用對應

## 文獻 1：MacQueen (1967)
> MacQueen, J. (1967). Some methods for classification and analysis of multivariate
> observations. *Proceedings of the Fifth Berkeley Symposium on Mathematical
> Statistics and Probability*, 1, 281–297.

**引用部分**：Lloyd 迭代式的 $k$-means 演算法定義——最小化群內平方和的目標函數
與「指派—更新質心」交替最佳化程序。本策略的分群階段直接採此標準形式
（`sklearn.cluster.KMeans`，`n_init=10`）。

**⚠ `ref/` 內尚無此文獻 PDF，請補充。**
可替代引用（`ref/` 內已有，皆將 $k$-means 用於配對交易選股）：
`2020-Enhancing a pairs trading strategy with the application of machine learning.pdf`、
`2021-Pairs Trading via Unsupervised Learning.pdf`。

## 文獻 2：Avellaneda & Lee (2010)／Hong & Hwang (2021)
> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities
> market. *Quantitative Finance*, 10(7), 761–782.　📄 `ref/2010-...pdf`
> Hong, G., & Hwang, S. (2021). In search of pairs using firm fundamentals.
> *European Journal of Finance*, 29(5).　📄 `ref/2021-In Search of Pairs...pdf`

**引用部分**：混合特徵的兩個來源——報酬 PCA 因子載荷作為價格行為表徵
（Avellaneda & Lee 第 2 節的因子模型），以及基本面相似性作為配對依據
（Hong & Hwang 的核心主張）。**此特徵矩陣與 HDBSCAN、Agglomerative 完全相同**，
是三者可比的前提。

## 文獻 3：Gatev et al. (2006)／Engle & Granger (1987)
> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading:
> Performance of a relative-value arbitrage rule. *RFS*, 19(3), 797–827.　📄
> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction.
> *Econometrica*, 55(2), 251–276.　📄

**引用部分**：群內排序採 Gatev 的最小平方距離（SSD）準則；
篩選層採 Engle-Granger 共整合檢定精神的 ADF 殘差檢定。
兩者於三種分群法間**參數完全一致**，故排序與篩選不構成混淆變因。


# 各階段行為

## 階段 1：混合特徵矩陣建構

與 Agglomerative、HDBSCAN 共用同一套特徵組裝流程，三區塊拼接：

- **價格行為（5 維）**：形成窗日報酬矩陣逐股標準化後取 PCA，
  以 $\sqrt{\text{特徵值}}$ 加權的因子載荷
- **公司基本面（2 維）**：$[\log(1+\text{MarketCap}),\ 1/PE]$，
  Point-in-Time 對齊（每窗取日期 ≤ 形成期末的最近一筆），
  缺值以同期產業中位數插補後 winsorize（1%–99%）
- **產業歸屬（12 維）**：11 個正規化 GICS 產業 + Unknown 的 one-hot

## 階段 2：區塊分別標準化與加權拼接

三區塊各自標準化後以等權（各 1.0）拼接為 19 維特徵向量。
分別標準化避免量綱差異使單一區塊主導歐氏距離。


## 階段 3：K-means 分群（依據：MacQueen 1967）

**目標函數**——最小化群內平方和：

$$\underset{S}{\arg\min} \sum_{j=1}^{k} \sum_{\mathbf{x} \in S_j}
\lVert \mathbf{x} - \boldsymbol{\mu}_j \rVert^2,
\qquad \boldsymbol{\mu}_j = \frac{1}{|S_j|}\sum_{\mathbf{x} \in S_j} \mathbf{x}$$

**Lloyd 迭代**（交替最佳化至收斂）：

1. **指派**：$S_j \leftarrow \{\mathbf{x}_i : \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert \le \lVert \mathbf{x}_i - \boldsymbol{\mu}_l \rVert\ \forall l\}$
2. **更新**：$\boldsymbol{\mu}_j \leftarrow$ 群 $S_j$ 的質心

**執行設定**：

- $k$ = 同窗 Agglomerative 的群數（見「群數如何決定」一節），並限制 $k \le N-1$
- `n_init = 10`：以 10 組不同初始質心各跑一次，取目標函數最小者
  （Lloyd 迭代只保證收斂至局部最佳）
- 固定 `random_state`，使同一窗口的分群結果可重現

**方法特性**：K-means 隱含假設群為**等向且大小相近的球狀結構**。
本策略的特徵空間含 12 維 one-hot 產業區塊，其幾何為離散頂點而非連續球狀——
此假設不匹配是 K-means 在矩陣中表現落後的預期原因。


## 階段 4：群內 SSD 排序與共整合篩選（依據：Gatev 2006／Engle-Granger 1987）

**排序**：於每個群內列舉所有配對，計算正規化對數價格的最小平方距離

$$SSD_{ij} = \sum_{t=1}^{T} \left( P'_{i,t} - P'_{j,t} \right)^2,
\qquad P'_{i,t} = \frac{\ln P_{i,t} - \mu_i^{form}}{\sigma_i^{form}}$$

依 $SSD$ 升序排列，逐一送入篩選層，填滿 `top_n` 為止。

**篩選**（三道，全數通過才納入）：

| 檢定 | 條件 | 意義 |
| :--- | :--- | :--- |
| ADF（`regression="n"`） | $p < 0.05$ | 價差序列定態，存在均衡關係 |
| OU 半衰期 | $1 \le -\ln 2 / \lambda \le 42$ | 回歸速度落在可交易區間 |
| Hurst 指數（R/S） | $H < 0.5$ | 序列具均值回歸而非趨勢特性 |

排序與篩選的所有參數與 HDBSCAN、Agglomerative **完全一致**。


## 階段 5–6：輸出與交易期銜接

輸出標準欄位 + `Sector_A/B`（真實 GICS）、`Cluster_ID_A/B`、`MarketCap_A/B`、`TrailingPE_A/B`。
排序流程不輸出 `OLS_Alpha`，交易期於標準化空間重建 spread（路徑 B，
見 `trading/zscore_trading.ipynb`）。形成期統計量整個交易期凍結，
基本面採 PIT 對齊，全流程無前視。

本策略配對另供 DRL 門檻選擇式交易端使用（見 `trading/drl_threshold_trading.ipynb`）。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依 SSD 升序取前幾組 |
| 價格因子數 | 5 | 特徵 | 價格行為區塊維度 |
| 三區塊權重 | 各 1.0 | 特徵 | 價格 / 基本面 / 產業 等權 |
| **群數 $k$** | **對齊同窗 Agglomerative** | **分組** | **確保三種分群法粒度可比** |
| 初始化重複次數 | 10 | 分組 | 取群內平方和最小者，緩解局部最佳 |
| 隨機種子 | 固定 | 分組 | 分群結果可重現 |
| 共整合顯著水準 | 0.05 | 篩選 | 群內 ADF 檢定門檻 |
| OU 半衰期區間 | [1, 42] 交易日 | 篩選 | 回歸速度可交易範圍 |
| 基本面資料源 | Point-in-Time 逐點 | 特徵 | 每窗取日期 ≤ 形成期末的最近一筆 |
